In [ ]:
# ============================================================
# Clean, de-identified script for conference submission
# Replace all personal paths with:
#   sys.path.append("/path/to/your/CausalICM/")
# Users must adjust this to the folder containing the CausalICM/functions package.
# ============================================================

import sys
sys.path.append("/path/to/your/CausalICM/")   # <-- USER MUST UPDATE THIS

import numpy as np
import pandas as pd
import copy
import GPy

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.kernel_ridge import KernelRidge as kernel_ridge
from sklearn.ensemble import RandomForestRegressor as rfr
from sklearn.linear_model import Ridge as ridge
from sklearn.metrics import mean_squared_error as mse

import matplotlib.pyplot as plt
from functools import partial
from scipy.special import expit

import statsmodels.api as sm
import statsmodels.formula.api as smf

# --- R interface
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri as np2ri
np2ri.activate()

# --- CausalICM imports
from CausalICM.functions.data_simulation_multivariate import data_simulation4
from CausalICM.functions.model_training import ICM


# ============================================================
# Helper utilities
# ============================================================
rmse = lambda x, y: np.sqrt(mse(x, y))

rfr_reg_parameter_pairs = [
    (
        rfr(),
        {
            "min_samples_leaf": list(range(1, 11, 3)),
            "n_estimators": [100, 200]
        }
    )
]

linear_reg_parameter_pairs = [
    (
        ridge(),
        {"alpha": np.logspace(-7, 7, 8), "tol": [1e-3]}
    )
]


def get_best_model_from_gridcv(X, Y, reg_parameter_pairs, print_flag=False, verbose=0):
    """
    Runs GridSearchCV over a list of (model, param-grid) pairs and returns the best model.
    """
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

    val_errs = []
    models = []

    for reg, parameters in reg_parameter_pairs:
        if print_flag:
            print("Trying:", str(reg)[:50])

        model = GridSearchCV(reg, parameters, cv=2, refit=True, verbose=verbose, n_jobs=10)
        model.fit(X, Y)

        val_errs.append(rmse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model.best_estimator_))

    min_ind = np.argmin(val_errs)
    return copy.deepcopy(models[min_ind])


# ============================================================
# Kallus Two-step GP Method
# ============================================================
def ite_adjusted_outcome(Y, T, _propensity, c=None):
    """
    Computes adjusted ITE using Kallus transformation.
    """
    if c is None:
        c = np.mean(Y)

    assert 0 < _propensity < 1

    out = np.zeros(Y.shape)
    out[T == 1] = (Y[T == 1] - c) / _propensity
    out[T == 0] = -(Y[T == 0] - c) / (1 - _propensity)
    return out


def run_methodGP(X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval):
    """
    Runs the two-step GP algorithm from Kallus.
    """

    # First-stage GP: observational control + treated
    kern = GPy.kern.RBF(input_dim=X_obs.shape[1], variance=1., lengthscale=1.)

    f0_gp = GPy.models.GPRegression(X_obs[T_obs == 0], Y_obs[T_obs == 0].reshape(-1, 1), kern.copy())
    f0_gp.optimize()

    f1_gp = GPy.models.GPRegression(X_obs[T_obs == 1], Y_obs[T_obs == 1].reshape(-1, 1), kern.copy())
    f1_gp.optimize()

    omega_eval_pred = f1_gp.predict(X_eval)[0] - f0_gp.predict(X_eval)[0]

    # Second stage: regress out bias η
    _propensity_rct = np.mean(T_rct)
    cate_rct_est = ite_adjusted_outcome(Y_rct, T_rct, _propensity=_propensity_rct)
    eta_rct_est = cate_rct_est.reshape(-1, 1) - (f1_gp.predict(X_rct)[0] - f0_gp.predict(X_rct)[0])

    best_eta_model = get_best_model_from_gridcv(
        X_rct.reshape(-1, X_rct.shape[1]), eta_rct_est, linear_reg_parameter_pairs
    )

    return best_eta_model, omega_eval_pred


# ============================================================
# Simulation Setup
# ============================================================
best_rho = pd.read_csv("sim4_best_rho.csv").values[0]

rmse_naiveGP_exp = np.zeros(100)
rmse_naiveGP_obs = np.zeros(100)
rmse_KallusGP = np.zeros(100)
rmse_ICM = np.zeros(100)

bias_naiveGP_exp = np.zeros(100)
bias_naiveGP_obs = np.zeros(100)
bias_KallusGP = np.zeros(100)
bias_ICM = np.zeros(100)

var_naiveGP_exp = np.zeros(100)
var_naiveGP_obs = np.zeros(100)
var_KallusGP = np.zeros(100)
var_ICM = np.zeros(100)

# Evaluation grid
test_data = np.round(
    np.c_[
        np.arange(-2.0, 2.04, 0.04),
        np.arange(-2.0, 2.04, 0.04),
        np.arange(-2.0, 2.04, 0.04),
        np.arange(-2.0, 2.04, 0.04),
        np.arange(-2.0, 2.04, 0.04)
    ]
)

# Storage for per-iteration results
bias_naive_exp_df = pd.DataFrame(test_data)
bias_naive_obs_df = pd.DataFrame(test_data)
bias_kallus_df = pd.DataFrame(test_data)
bias_icm_df = pd.DataFrame(test_data)

all_data = []


# ============================================================
# Main Simulation Loop
# ============================================================
for i in range(100):
    print(f"Iteration {i+1}")

    # --- Simulate data
    data_exp, data_obs = data_simulation4(1000, beta0=-10.0, beta1=-8.0, beta2=-8.0)

    df_exp = pd.DataFrame(data_exp, columns=["S", "X1", "X2", "X3", "X4", "X5", "A", "Y"])
    df_exp["datasetNo"] = i + 1
    df_obs = pd.DataFrame(data_obs, columns=["S", "X1", "X2", "X3", "X4", "X5", "A", "Y"])
    df_obs["datasetNo"] = i + 1

    all_data.append(pd.concat([df_exp, df_obs], ignore_index=True))

    # --- Extract data by treatment group
    X0_obs = data_obs[:, 1:6][data_obs[:, 6] == 0]
    X1_obs = data_obs[:, 1:6][data_obs[:, 6] == 1]
    Y0_obs = data_obs[:, 7][data_obs[:, 6] == 0]
    Y1_obs = data_obs[:, 7][data_obs[:, 6] == 1]

    X0_exp = data_exp[:, 1:6][data_exp[:, 6] == 0]
    X1_exp = data_exp[:, 1:6][data_exp[:, 6] == 1]
    Y0_exp = data_exp[:, 7][data_exp[:, 6] == 0]
    Y1_exp = data_exp[:, 7][data_exp[:, 6] == 1]

    # True CATE
    true_cate = 1 + test_data[:, 0] + test_data[:, 1] + test_data[:, 0] ** 2 + test_data[:, 1] ** 2

    # ============================================================
    # Naive GP – observational
    # ============================================================
    kern = GPy.kern.RBF(input_dim=5)
    m0_obs = GPy.models.GPRegression(X0_obs, Y0_obs.reshape(-1, 1), kern.copy())
    m1_obs = GPy.models.GPRegression(X1_obs, Y1_obs.reshape(-1, 1), kern.copy())
    m0_obs.optimize()
    m1_obs.optimize()

    naiveCATE_obs = m1_obs.predict(test_data)[0] - m0_obs.predict(test_data)[0]

    # ============================================================
    # Naive GP – experimental
    # ============================================================
    m0_exp = GPy.models.GPRegression(X0_exp, Y0_exp.reshape(-1, 1), kern.copy())
    m1_exp = GPy.models.GPRegression(X1_exp, Y1_exp.reshape(-1, 1), kern.copy())
    m0_exp.optimize()
    m1_exp.optimize()

    naiveCATE_exp = m1_exp.predict(test_data)[0] - m0_exp.predict(test_data)[0]

    # ============================================================
    # Kallus GP
    # ============================================================
    eta_model, omega_eval_pred = run_methodGP(
        data_exp[:, 1:6], data_exp[:, 7], data_exp[:, 6],
        data_obs[:, 1:6], data_obs[:, 7], data_obs[:, 6],
        test_data
    )
    tau_pred_eval = eta_model.predict(test_data) + omega_eval_pred

    # ============================================================
    # ICM
    # ============================================================
    m0, m1 = ICM(
        X_E=data_exp[:, 1:6], X_O=data_obs[:, 1:6],
        T_E=data_exp[:, 6], T_O=data_obs[:, 6],
        Y_E=data_exp[:, 7], Y_O=data_obs[:, 7],
        r=2, ID=5, AD=[0, 1, 2, 3, 4], rho=best_rho
    )

    predY0 = m0.predict_noiseless(np.c_[test_data, np.zeros(test_data.shape[0])])[0]
    predY1 = m1.predict_noiseless(np.c_[test_data, np.zeros(test_data.shape[0])])[0]
    predCATE = predY1 - predY0

    # ============================================================
    # Compute Metrics
    # ============================================================
    rmse_ICM[i] = rmse(predCATE, true_cate)
    rmse_KallusGP[i] = rmse(tau_pred_eval, true_cate)
    rmse_naiveGP_exp[i] = rmse(naiveCATE_exp, true_cate)
    rmse_naiveGP_obs[i] = rmse(naiveCATE_obs, true_cate)

    bias_naiveGP_exp[i] = np.mean(naiveCATE_exp - true_cate)
    bias_naiveGP_obs[i] = np.mean(naiveCATE_obs - true_cate)
    bias_KallusGP[i] = np.mean(tau_pred_eval - true_cate)
    bias_ICM[i] = np.mean(predCATE - true_cate)

    var_naiveGP_exp[i] = np.var(naiveCATE_exp)
    var_naiveGP_obs[i] = np.var(naiveCATE_obs)
    var_KallusGP[i] = np.var(tau_pred_eval)
    var_ICM[i] = np.var(predCATE)

    bias_naive_exp_df[f"Iteration_{i+1}"] = naiveCATE_exp - true_cate
    bias_naive_obs_df[f"Iteration_{i+1}"] = naiveCATE_obs - true_cate
    bias_kallus_df[f"Iteration_{i+1}"] = tau_pred_eval - true_cate
    bias_icm_df[f"Iteration_{i+1}"] = predCATE - true_cate


# ============================================================
# Save Results
# ============================================================
pd.concat(all_data, ignore_index=True).to_csv("datasets_sim4_mult.csv", index=False)

sim4_modelComparison_RMSEs_df = pd.DataFrame(
    np.c_[
        rmse_naiveGP_exp, rmse_naiveGP_obs, rmse_KallusGP, rmse_ICM,
        bias_naiveGP_exp, bias_naiveGP_obs, bias_KallusGP, bias_ICM,
        var_naiveGP_exp, var_naiveGP_obs, var_KallusGP, var_ICM
    ],
    columns=[
        "RMSE_naiveGPexp", "RMSE_naiveGPobs", "RMSE_KallusGP", "RMSE_ICM",
        "bias_naiveGPexp", "bias_naiveGPobs", "bias_KallusGP", "bias_ICM",
        "var_naiveGPexp", "var_naiveGPobs", "var_KallusGP", "var_ICM"
    ]
)
sim4_modelComparison_RMSEs_df.to_csv("sim4_modelComparison.csv", index=False)

bias_naive_exp_df.to_csv("bias_naive_exp.csv", index=False)
bias_naive_obs_df.to_csv("bias_naive_obs.csv", index=False)
bias_kallus_df.to_csv("bias_kallus.csv", index=False)
bias_icm_df.to_csv("bias_icm.csv", index=False)
